In [1]:
import sys
sys.path.append('..')
from osp import *
import html

In [2]:
df_feats = STASH_SLICE_FEATS.df
df_feats

,pos_DT,pos_VBZ,pos_RB,pos_JJ,pos_IN,pos_NN,pos_PRP$,pos_NNS,pos_TO,pos_VB,...,pos_LS,pos_NFP,deprel_vocative,pos_SYM,deprel_orphan,pos_AFX,pos_ADD,deprel_goeswith,pos_GW,deprel_obl:tmod
_key,,,,,,,,,,,,,,,,,,,,,
phil/10.2307/2380200__04,84.104289,46.257359,81.581161,116.904962,111.017662,102.607233,15.979815,45.416316,22.708158,58.031960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/2514892__02,87.415222,20.346647,16.578749,97.211756,143.180106,113.790505,5.275057,66.314996,5.275057,11.303693,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/14710__03,102.811245,11.244980,46.586345,100.401606,130.120482,167.068273,12.048193,52.208835,8.032129,17.670683,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/20111773__02,98.193244,56.559309,56.559309,98.193244,135.113904,109.190888,6.284368,35.349568,9.426551,24.351925,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/2379466__05,91.644205,32.345013,65.588500,79.065588,139.263252,141.958670,11.680144,53.908356,12.578616,40.431267,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lit/468421__06,116.161616,40.404040,44.612795,75.757576,136.363636,162.457912,11.784512,54.713805,14.309764,37.878788,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/29782027__24,80.839895,18.372703,29.921260,51.968504,70.341207,87.664042,0.524934,25.721785,7.349081,25.721785,...,NaN,5.249344,NaN,0.524934,NaN,NaN,NaN,NaN,NaN,NaN
lit/459531__04,92.471358,40.098200,36.824877,80.196399,135.842881,134.206219,18.003273,49.099836,21.276596,32.733224,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
feats = ['deprel_cop', 'pos_MD', 'deprel_aux', 'deprel_expl']
top_slices = df_feats[feats].mean(axis=1).sort_values(ascending=False)
top_slice = top_slices.index[0]
top_slice

'phil/10.2307/4544720__03'

In [2]:
doc = stanza.Document.from_serialized(STASH_SLICES_NLP[top_slice])

NameError: name 'top_slice' is not defined

In [3]:
# sent = doc.sentences[0]
sent = get_nlp()("The world is all that is the case.").sentences[0]

In [10]:
def get_space_after(sent):
    o=[]
    for w in sent.tokens:
        wd = w.to_dict()[0]
        o.append('SpaceAfter=No' in wd.get('misc',''))
    return o


def get_sent_html_simple(
    sent,
    feat_bold=None,
    feat_italic=None,
    feat_underline=None,
    show_clause_labels=False,
    list_tag='ul',
):
    if list_tag not in {'ul', 'ol'}:
        raise ValueError("list_tag must be 'ul' or 'ol'")

    clause_df = get_syntax_df(sent).copy()
    clause_df['space_after'] = get_space_after(sent)

    def _normalize_spec(spec):
        if spec is None:
            return set(), set()
        if isinstance(spec, dict):
            dep = spec.get('deprel', [])
            pos = spec.get('pos', [])
            dep = {dep} if isinstance(dep, str) else set(dep)
            pos = {pos} if isinstance(pos, str) else set(pos)
            return dep, pos
        values = {spec} if isinstance(spec, str) else set(spec)
        dep = set()
        pos = set()
        for v in values:
            if isinstance(v, str) and v.startswith('deprel_'):
                dep.add(v[len('deprel_'):])
            elif isinstance(v, str) and v.startswith('pos_'):
                pos.add(v[len('pos_'):])
            else:
                dep.add(v)
                pos.add(v)
        return dep, pos

    def _matches(row, spec):
        dep_set, pos_set = _normalize_spec(spec)
        return (row.word_deprel in dep_set) or (row.word_pos in pos_set)

    def _style_word(row):
        txt = html.escape(str(row.word))
        if _matches(row, feat_underline):
            txt = f"<u>{txt}</u>"
        if _matches(row, feat_italic):
            txt = f"<i>{txt}</i>"
        if _matches(row, feat_bold):
            txt = f"<b>{txt}</b>"
        if not row.space_after:
            txt += ' '
        return txt

    rows = clause_df.sort_values('word_i').itertuples(index=False)

    segments = []
    active_clause_id = None
    for row in rows:
        clause_id = active_clause_id if row.word_deprel == 'punct' else row.clause_id
        if clause_id is None:
            clause_id = row.clause_id

        if not segments or segments[-1]['clause_id'] != clause_id:
            segments.append(
                {
                    'clause_id': clause_id,
                    'depth': max(0, int(row.clause_depth)),
                    'label': 'IC' if row.clause_type == 'main' else 'DC',
                    'words': [],
                    'children': [],
                }
            )
        segments[-1]['words'].append(_style_word(row))
        active_clause_id = clause_id

    if not segments:
        return f'<{list_tag}></{list_tag}>'

    root = {'children': []}
    stack = [root]  # stack length = current depth + 1 (root)

    for seg in segments:
        depth = seg['depth']
        while len(stack) > depth + 1:
            stack.pop()
        node = {
            'label': seg['label'],
            'depth': depth,
            'text': ''.join(seg['words']),
            'children': [],
        }
        stack[-1]['children'].append(node)
        stack.append(node)

    def _render(nodes):
        out = [f'<{list_tag}>']
        for node in nodes:
            out.append('<li>')
            if show_clause_labels:
                out.append(f"{node['label']}d{node['depth']}: ")
            out.append(node['text'])
            if node['children']:
                out.append(_render(node['children']))
            out.append('</li>')
        out.append(f'</{list_tag}>')
        return ''.join(out)

    return _render(root['children'])


def get_sents_html_simple(
    sents,
    feat_bold=None,
    feat_italic=None,
    feat_underline=None,
    show_clause_labels=False,
    sentence_list_tag='ol',
    clause_list_tag='ol',
):
    if sentence_list_tag not in {'ul', 'ol'}:
        raise ValueError("sentence_list_tag must be 'ul' or 'ol'")
    if clause_list_tag not in {'ul', 'ol'}:
        raise ValueError("clause_list_tag must be 'ul' or 'ol'")

    out = [f'<{sentence_list_tag}>']
    for sent in sents:
        out.append('<li>')
        out.append(
            get_sent_html_simple(
                sent,
                feat_bold=feat_bold,
                feat_italic=feat_italic,
                feat_underline=feat_underline,
                show_clause_labels=show_clause_labels,
                list_tag=clause_list_tag,
            )
        )
        out.append('</li>')
    out.append(f'</{sentence_list_tag}>')
    return ''.join(out)

In [25]:
doc = get_nlp()("The world is all that is the case, and it is not at all what is not the case. It really is!").sentences

In [27]:
htmlx=get_sents_html_simple(doc, feat_bold='deprel_cop')
print(htmlx)
HTML(htmlx)

<ol><li><ol><li>The world <b>is</b> all <ol><li>that <b>is</b> the case, </li></ol></li><li>and it <b>is</b> not at all <ol><li>what <b>is</b> not the case. </li></ol></li></ol></li><li><ol><li>It really is!</li></ol></li></ol>


In [28]:
print(html_to_latex(htmlx).replace('\n\n','\n'))

\begin{enumerate}
\item
\begin{enumerate}
\item The world \textbf{is} all
\begin{enumerate}
\item that \textbf{is} the case,
\end{enumerate}
\item and it \textbf{is} not at all
\begin{enumerate}
\item what \textbf{is} not the case.
\end{enumerate}
\end{enumerate}
\item
\begin{enumerate}
\item It really is!
\end{enumerate}
\end{enumerate}


In [9]:
htmlx=get_sent_html_simple(sent, feat_bold='deprel_cop', list_tag='ol')
htmlx

'<ol><li>The world <b>is</b> all <ol><li>that <b>is</b> the case.</li></ol></li></ol>'

In [8]:


HTML(htmlx)

In [83]:
print(html_to_latex(htmlx))

\begin{itemize}
\item It appears to me
\begin{itemize}
\item that in Ethics, as in all other philosophical studies, the difficulties and disagreements,
\begin{itemize}
\item of which its history \textbf{is} full,

\end{itemize}
\item \textbf{are} mainly due to a very simple cause: namely to the attempt
\begin{itemize}
\item to answer questions,
\item without first discovering precisely what question
\begin{itemize}

\begin{itemize}
\item it

\end{itemize}

\end{itemize}
\item \textbf{is}
\begin{itemize}

\begin{itemize}
\item which

\end{itemize}
\item you desire
\begin{itemize}
\item to answer.

\end{itemize}

\end{itemize}

\end{itemize}

\end{itemize}

\end{itemize}


In [48]:
for w in sent.tokens:
    print(w.to_dict())

[{'id': 1, 'text': 'The', 'lemma': 'the', 'upos': 'DET', 'xpos': 'DT', 'feats': 'Definite=Def|PronType=Art', 'head': 2, 'deprel': 'det', 'start_char': 0, 'end_char': 3, 'ner': 'O', 'multi_ner': ('O',)}]
[{'id': 2, 'text': 'world', 'lemma': 'world', 'upos': 'NOUN', 'xpos': 'NN', 'feats': 'Number=Sing', 'head': 3, 'deprel': 'nsubj', 'start_char': 4, 'end_char': 9, 'ner': 'O', 'multi_ner': ('O',)}]
[{'id': (3, 4), 'text': 'cannot', 'start_char': 10, 'end_char': 16, 'ner': 'O', 'multi_ner': ('O',), 'misc': 'SpaceAfter=No'}, {'id': 3, 'text': 'can', 'lemma': 'can', 'upos': 'AUX', 'xpos': 'MD', 'feats': 'VerbForm=Fin', 'head': 0, 'deprel': 'root', 'start_char': 10, 'end_char': 13}, {'id': 4, 'text': 'not', 'lemma': 'not', 'upos': 'PART', 'xpos': 'RB', 'feats': 'Polarity=Neg', 'head': 3, 'deprel': 'advmod', 'start_char': 13, 'end_char': 16}]
[{'id': 5, 'text': ',', 'lemma': ',', 'upos': 'PUNCT', 'xpos': ',', 'head': 3, 'deprel': 'punct', 'start_char': 16, 'end_char': 17, 'ner': 'O', 'multi_ne

In [ ]:
get_syntax_df(sent).word.tolist()

In [ ]:
HTML(get_sent_html(sent))

In [ ]:
def show_slice(slice_id, feats=[]):
    doc = stanza.Document.from_serialized(STASH_SLICES_NLP[slice_id])

    
    return doc

In [ ]:
show_slice(top_slice)